# Optimización Logística en E-Commerce
### Exploracion, limpieza y organizacion de los datos

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path
import os

# Definicion de rutas estandarizadas
BASE_DIR = Path.cwd().parent
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

PATH_ORDERS = RAW_DIR / "olist_orders_dataset.csv"
PATH_ORDER_ITEMS = RAW_DIR / "olist_order_items_dataset.csv"
PATH_ORDER_REVIEWS = RAW_DIR / "olist_order_reviews_dataset.csv"
PATH_NAME_TRANSLATIONS = RAW_DIR / "product_category_name_translation.csv"
PATH_CUSTOMERS = RAW_DIR / "olist_customers_dataset.csv"
PATH_PRODUCTS = RAW_DIR / "olist_products_dataset.csv"

In [2]:
# Carga de datasets en memoria
df_orders = pd.read_csv(PATH_ORDERS)
df_order_items = pd.read_csv(PATH_ORDER_ITEMS)
df_order_reviews = pd.read_csv(PATH_ORDER_REVIEWS)
df_name_translations = pd.read_csv(PATH_NAME_TRANSLATIONS)
df_customers = pd.read_csv(PATH_CUSTOMERS)
df_products = pd.read_csv(PATH_PRODUCTS)

In [3]:
# Consolidacion del DataFrame maestro
# Se utiliza 'inner' para entidades dependientes y 'left' para atributos opcionales
df_final = df_orders.merge(df_order_items, on='order_id', how='inner') \
    .merge(df_customers, on='customer_id', how='inner') \
    .merge(df_products, on='product_id', how='inner') \
    .merge(df_order_reviews, on='order_id', how='left') \
    .merge(df_name_translations, on='product_category_name', how='left')

In [4]:
# Estandarizacion de formatos de fecha
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    df_final[col] = pd.to_datetime(df_final[col], format='%Y-%m-%d %H:%M:%S')

In [5]:
# Ingenieria de caracteristicas para metricas de negocio
df_final['delivery_delay_days'] = (df_final['order_delivered_customer_date'] - df_final['order_estimated_delivery_date']).dt.days

In [6]:
# Vectorizacion para categorizacion de reseñas (mayor rendimiento que lambda)
# Maneja correctamente los valores nulos derivados del left merge previo
df_final['review_score_category'] = np.where(
    df_final['review_score'].isna(), 
    'no_review', 
    np.where(df_final['review_score'] <= 2, 'negative', 'positive')
)

In [7]:
# Aislamiento del subset de datos especifico para generacion de graficos
target_columns = [
    'order_id', 'customer_id', 'product_id', 
    'product_category_name_english', 'order_purchase_timestamp', 
    'order_delivered_customer_date', 'order_estimated_delivery_date', 
    'customer_city', 'customer_state', 'delivery_delay_days', 
    'review_score_category'
]

df_preguntas = df_final[target_columns].copy()
df_preguntas.head(10)

,order_id,customer_id,product_id,product_category_name_english,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,customer_city,customer_state,delivery_delay_days,review_score_category
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,87285b34884572647811a353c7ac498a,housewares,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,sao paulo,SP,-8.0,positive
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,595fac2a385ac33a80bd5114aec74eb8,perfumery,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,barreiras,BA,-6.0,positive
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,aa4383b373c6aca5d8797843e5594415,auto,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,vianopolis,GO,-18.0,positive
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,d0b61bfb1de832b15ba9d266ca96e5b0,pet_shop,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,sao goncalo do amarante,RN,-13.0,positive
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,65266b2da20d04dbe00c5c2d3bb7859e,stationery,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,santo andre,SP,-10.0,positive
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,060cb19345d90064d1015407193c233d,auto,2017-07-09 21:57:05,2017-07-26 10:57:55,2017-08-01,congonhinhas,PR,-6.0,positive
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,a1804276d9941ac0733cfd409f5206eb,NaN,2017-04-11 12:22:08,NaT,2017-05-09,santa rosa,RS,NaN,negative
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,4520766ec412348b8d4caa5e8a18c464,auto,2017-05-16 13:10:30,2017-05-26 12:55:51,2017-06-07,nilopolis,RJ,-12.0,positive
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,ac1789e492dcd698c5c10b97a671243a,furniture_decor,2017-01-23 18:29:09,2017-02-02 14:08:10,2017-03-06,faxinalzinho,RS,-32.0,negative
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,9a78fb9862b10749a117f7fc3c31f051,office_furniture,2017-07-29 11:55:02,2017-08-16 17:14:30,2017-08-23,sorocaba,SP,-7.0,positive


In [ ]:
# Guardado del DataFrame procesado en formato parquet para optimizacion de espacio y velocidad de lectura
df_preguntas.to_parquet(PROCESSED_DIR / "ecommerce_analytical_base.parquet", index=False)